<div style="text-align: center">
<img src="https://github.com/LinkedEarth/Logos/blob/master/PyleoTUPS/pyleotups_logo.png?raw=true" alt="PyleoTUPS logo" width="400">
</div>

# Working with NOAA NCEI for Paleoclimatology

## Authors

Deborah Khider [![ORCID](https://img.shields.io/badge/ORCID-0000--0001--7501--8430-A6CE39?logo=orcid)](https://orcid.org/0000-0001-7501-8430), Dhiren Oswal [![ORCID](https://img.shields.io/badge/ORCID-0009--0001--2495--2626-A6CE39?logo=orcid)](https://orcid.org/0009-0001-2495-2626)


## Preamble

The goal of this tutorial is to familiarize you with the `NOAADataset` object and its functionalities in PyleoTUPS. In this tutorial, we focus specifically on searching by NOAA study ID; a separate tutorial will cover the full range of NOAA search capabilities.

In the NOAA framework, study IDs represent a *publication unit* for a dataset—that is, all data associated with a given study. For example, a study may include multiple data tables corresponding to different sites. While these tables share the same study ID, each site is assigned a unique site ID.


### Goals

 - Search for datasets using a specific study ID
 - Obtain information such as location, publication, variable metadata and the associated data.

### Pre-requisites

- Understanding of NOAA datasets

### Reading time

15 min

Let's import our packages!

In [1]:
import pyleotups as pt
import pandas as pd

## Accessing a dataset using the NOAA study ID

This is the most basic search you can perform, but if you know which dataset you are looking for, it is the easiest way to get all the needed information. First, you need to create a `NOAADataset` object that will store the information.

In [2]:
ds=pt.NOAADataset()

Let's do a simple search, knowing the NOAA study ID. For this example, let's use the dataset from [Clemens et al. (2021)](https://doi.org/10.1126/sciadv.abg3848), which can be accessed through [NOAA Paleo portal](https://www.ncei.noaa.gov/access/paleo-search/study/33213). The study contains several tables of measurements made on a marine sediment at site IODP U1446:

<div style="
    padding: 10px; 
    background-color: #fff3cd; 
    border-left: 6px solid #ffcc00; 
    margin: 10px 0;">
    <strong>Warning:</strong> Even for one study, the search may take some time. 
</div>

In [3]:
res = ds.search_studies(noaa_id=33213)

[2026-04-06 12:46:18,879][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).
[2026-04-06 12:46:18,881][INFO] - search_studies: Input Query includes geographical bounds. Inspect the results to ensure they match your intended region as one study can contain sites across various parts of the world.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&NOAAStudyId=33213&limit=100


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 1911.72it/s]
[2026-04-06 12:46:19,473][INFO] - Retrieved 1 studies.


The `summary` method provides basic information about the dataset, such as the name of the study, the NOAA [DataType](https://www.ncei.noaa.gov/products/paleoclimatology), the time coverage and the associated publication. The function retuns a `pandas.DataFrame`.

In [4]:
df_summary = ds.get_summary()
display(df_summary)

,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,33213,74834,"Bay of Bengal, Northeast Indian Margin Stable ...",PALEOCEANOGRAPHY,1462580,280,-1460630,1670,"(19.083, 19.083, 85.733, 85.733)","Provided Keywords: Indian monsoon, South Asian...",None,"Steven Clemens, Masanobu Yamamoto, Kaustubh Th...","[{'Author': 'Clemens, Steven; Yamamoto, Masano...","[[{'DataTableID': '45857', 'DataTableName': 'U...",[{'fundingAgency': 'US National Science Founda...


Let's have a look at the information returned in this DataFrame:

* `StudyID`: Matches the value we searched for. While redundant in this case, it becomes useful when querying using other criteria.
* `XMLID`: A different identifier for the same study. Each `StudyID` is associated with a corresponding `XMLID`.
* `Study Name`: The name of the study.
* `DataType`: Uses NOAA’s nomenclature for the data type.
* `EarliestYearBP`, `MostRecentYearBP`, `EarliestYearCE`, `MostRecentYearCE`: The temporal bounds of the dataset.
* `Coverage`: The geographic area covered by the study. In this example, it corresponds to a single point. This information can also be retrieved using the `get_geo` function.
* `StudyNotes`: Includes descriptive notes and keywords associated with the study.
* `ScienceKeywords`: Standardized science keywords.
* `Investigators`: The contributors to the study.
* `Publications`: Information about publications associated with the dataset. This can also be retrieved using the `get_publication` function.
* `Sites`: The sites associated with the NOAA study. A single study may include multiple sites, and each site may contain multiple data tables. This information can be retrieved using the `get_sites` function.
* `Funding`: Information about funding sources for the study. This can be retrieved using the `get_funding` function.

This function provides a high-level overview of the study and corresponds to the information available on the [NOAA landing page](https://www.ncei.noaa.gov/access/paleo-search/study/33213).

To facilitate downstream analysis, we provide additional functions that extract key components of the study into separate `Pandas.DataFrame` objects, making it easier to work with specific aspects of the data.

### Obtaining Information about Funding and Publications

To get more details about the funding:

In [5]:
df_funding = ds.get_funding()
display(df_funding)

,StudyID,StudyName,FundingAgency,FundingGrant
0,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",US National Science Foundation,OCE1634774
1,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",Japan Society for the Promotion of Science (JSPS),JPMXS05R2900001
2,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",UK Natural Environment Research Council (NERC),"JPMXS05R2900001, 19H05595"
3,33213,"Bay of Bengal, Northeast Indian Margin Stable ...",United States Geological Survey (USGS),NE/L002493/1


Note that each table has a studyID key. You can think of this as the key in relational databases. It may not matter right now, but this will become useful as you use more advanced search functionalities which will return several studies.

In addition to funding information, `PyleoTUPS` allows you to get information about the publication associated with the dataset. 

In [6]:
bib, df_pub = ds.get_publications()
display(df_pub)

,Author,Title,Journal,Year,Volume,Number,Pages,Type,DOI,URL,CitationKey,StudyID,StudyName
0,"Clemens, Steven; Yamamoto, Masanobu; Thirumala...",Remote and Local Drivers of Pleistocene South ...,Science Advances,2021,7,23,NaN,publication,10.1126/sciadv.abg3848,http://dx.doi.org/10.1126/sciadv.abg3848,M._Remote_2021_33213,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."


<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> You can set the `save` parameter to True if you want to save a copy of the BibTeX entries.
</div>

### Obtaining Information about Geographical Location

Similarly, you can return information about the location of the record: 

In [7]:
df_geo = ds.get_geo()
display(df_geo)

,StudyID,DataType,SiteID,SiteName,LocationName,Latitude,Longitude,MinElevation,MaxElevation
0,33213,PALEOCEANOGRAPHY,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440


When getting the geographical coordinates for a study, each row in the returned DataFrame will correspond to a specific site. In this case, there is only one site in the study. We will be looking at a more interesting case later in this tutorial. 

### Obtaining Variable Information

The next step is to actually obtain the data tables present in the dataset:

In [9]:
df_tables = ds.get_tables()
display(df_tables)

,DataTableID,DataTableName,TimeUnit,FileURL,Variables,FileDescription,TotalFilesAvailable,SiteID,SiteName,LocationName,Latitude,Longitude,MinElevation,MaxElevation,StudyID,StudyName
0,45857,U1446 Benthic Isotopes Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, Hole, Type, Section, Core, Section_Dept...",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
1,45858,U1446 Planktic Isotopes Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, Hole, Type, Section, Comment, Core, Sec...",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
2,45859,U1446 TEX86H_SST Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, Hole, Type, Section, Core, Section_Dept...",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
3,45860,U1446 d18Osw Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Age, SL_Scaled, SL_Scaled_averaged, d18O_G_ru...",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
4,45861,U1446 LeafWax CarbonIsotope Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, Hole, Type, Section, Core, Section_Dept...",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
5,45862,U1446 Mg/Ca Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, Hole, Type, Analytical_Facility, Core, ...",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
6,45863,U1446 Rb/Ca Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, Hole, Type, Section, Core, Section_Dept...",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."
7,45864,U1446 Age Model Clemens2021,cal yr BP,https://www.ncei.noaa.gov/pub/data/paleo/contr...,"[Site, Comments, Sample_Depth, Age]",NOAA Template File,1,58697,IODP U1446,Ocean>Indian Ocean,19.083,85.733,-1440,-1440,33213,"Bay of Bengal, Northeast Indian Margin Stable ..."


Note that each `TableID` is unique and can be used to get more information about the data. Most of the fields returned here are self-explanatory, but let's go over the few that require further explanation and design consideration:
* `fileURL`: This is where the actual data lives! So far, all the information we have gathered has been made available through the NOAA API response in a JSON file. Now it is time to actually read the data!
* `FileDescription`: This is an internal representation for NOAA and does inform about the type of files to be found at the `fileURL`. PyleoTUPS can only read text files, which means file description such as NOAA Template File, NOAA Template File - Sunthesis Metadata, Raw Measurements - NOAA Template File, Chronology - NOAA Template File. However, PyleoTUPS will not work on other format such as NetCDF (we recommend the [`xarray` library](https://xarray.dev) for these files), `.lpd` (we have created a [library to deal with LiPD files called `PyLiPD`](https://pylipd.readthedocs.io/en/latest/)), html, or pickle file.

<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> Let's make matters more complicated! The same TableID can be used multiple times for different file format. For instance, if the data uses the NOAA template and LiPD file format. AND a TableID (which is really the file) can actually have multiple tables in them.
</div>

According to the table above, our unique study contains one unique site (SiteID = 58697) and eight tables. To get the data, you need to pass the `DataTableID` or `FileURL` to the `get_data()` method. Let's have a look at the TEX86 data:

In [11]:
dfs = ds.get_data(dataTableIDs="45859")
type(dfs)

list

Notice that we are returning a list. This is because in some cases, the PyleoTUPS parser may identify more than one table in the file. This is most common with older studies. 

Let's have a look at the first and (only) table:

In [12]:
display(dfs[0])

,Site,Hole,Core,Type,Section,Section_Depth,Sample_Depth,Age,TEX86H,SST
0,U1446,C,1,H,1,4.5,0.045,0.31,-0.1177,30.55
1,U1446,C,1,H,1,31.5,0.315,0.66,-0.1216,30.28
2,U1446,C,1,H,1,61.5,0.615,1.04,-0.1183,30.51
3,U1446,C,1,H,1,90.5,0.905,1.41,-0.1089,31.15
4,U1446,C,1,H,1,121.5,1.215,1.80,-0.1155,30.70
...,...,...,...,...,...,...,...,...,...,...
605,U1446,C,23,F,4,7,203.365,1458.08,-0.1535,28.10
606,U1446,C,23,F,4,37,203.665,1459.49,-0.1640,27.38
607,U1446,C,23,F,4,67,203.965,1460.90,-0.1592,27.71
608,U1446,C,23,F,4,97,204.265,1462.32,-0.1649,27.32


And we have the data! How about more metadata about the variables themselves?

Well, there is a function for this as well:

In [13]:
df_var = ds.get_variables(dataTableIDs="45859")
display(df_var)

,StudyID,SiteID,FileURL,VariableName,cvDataType,cvWhat,cvMaterial,cvError,cvUnit,cvSeasonality,cvDetail,cvMethod,cvAdditionalInfo,cvFormat,cvShortName
DataTableID,,,,,,,,,,,,,,,
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Site,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,Site identification,Character,Site
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Hole,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,Hole drilled at Site U1446,Character,Hole
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Type,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,H (9 m hydraulic piston core) F (4.5 m hydrau...,Character,Type
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Section,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,Section number ( 1 through 7 and core catcher ...,Character,Section
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Core,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,NaN,None,None,NaN,Core number,Numeric,Core
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Section_Depth,PALEOCEANOGRAPHY,sampling metadata>sample identification,NaN,None,length unit>centimeter,None,None,NaN,Mid-depth of sample within section,Numeric,Section_Depth
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Sample_Depth,CLIMATE RECONSTRUCTIONS|PALEOCEANOGRAPHY,depth variable>depth,NaN,None,length unit>meter,None,None,NaN,Spliced core composite depth below sea floor ...,Numeric,Sample_Depth
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,Age,CLIMATE RECONSTRUCTIONS|PALEOCEANOGRAPHY,age variable>age,NaN,None,time unit>age unit>calendar kiloyear before pr...,None,None,NaN,NaN,Numeric,Age
45859,33213,58697,https://www.ncei.noaa.gov/pub/data/paleo/contr...,TEX86H,PALEOCEANOGRAPHY,chemical composition>compound>organic compound...,NaN,None,dimensionless,None,None,laboratory method>chromatography>liquid chroma...,index of Schouten et al. 2002; extracted lipids,Numeric,TEX86H


For more information about the meaning of the columns and possible values, please have a look at the [NOAA PaST Thesaurus](https://www.ncei.noaa.gov/products/paleoclimatology/paleoenvironmental-standard-terms-thesaurus). 

<div style="border-left: 4px solid #1f77b4; background: #f0f8ff; padding: 0.5em 1em; border-radius: 4px;">
  <strong>Note:</strong> You can pass multiple tables IDs as a list. The function will always return a list of DataFrames (hence why we selected the first one in the code above.)
</div>

Some relevant metadata for each column is also stored in the DataFrame attributes:

In [14]:
dfs[0].attrs

{'variables': ['Site',
  'Hole',
  'Core',
  'Type',
  'Section',
  'Section_Depth',
  'Sample_Depth',
  'Age',
  'TEX86H',
  'SST'],
 'NOAAStudyId': '33213',
 'StudyName': 'Bay of Bengal, Northeast Indian Margin Stable Isotope, Biomarker and SST Reconstructions since the Mid-Pleistocene'}

Instead of the table ID, you can also pass the file URL to get the data:

In [16]:
df1 = ds.get_data(file_urls="https://www.ncei.noaa.gov/pub/data/paleo/contributions_by_author/clemens2021/clemens2021-u1446-mgca-noaa.txt")[0]
display(df1.head())

,Site,Hole,Core,Type,Section,Section_Depth,Sample_Depth,Age,Mg/Ca,Analytical_Facility,Mg/Ca_SST
0,U1446,C,1,H,1,5,0.05,0.32,4.63,Rosenthal (Rutgers),28.41
1,U1446,C,1,H,1,32,0.32,0.66,4.62,Rosenthal (Rutgers),28.46
2,U1446,C,1,H,1,62,0.62,1.05,4.72,Rosenthal (Rutgers),28.43
3,U1446,C,1,H,1,91,0.91,1.42,4.58,Rosenthal (Rutgers),28.63
4,U1446,C,1,H,1,122,1.22,1.81,4.70,Rosenthal (Rutgers),28.61


You can choose the method that is more convenient for you. 



Let's have a look at some more complicated cases

## Dataset with Multiple Sites

As mentioned, some studies may have multiple sites. Let's see how this affect the functionalities. For this example, let's have a look a the temperature reconstruction for the Greater Yellowstone Ecoregion by [King et al. (2021)](htpps://doi.org/10.1029/2020GL092269), which can be found [here](https://www.ncei.noaa.gov/access/paleo-search/study/32833).

In [17]:
ds2 = pt.NOAADataset()

In [19]:
res = ds2.search_studies(noaa_id = 32833)
display(res)

[2026-04-06 14:11:19,202][INFO] - search_studies: Limit defaulted to 100 (PyleoTUPS).
[2026-04-06 14:11:19,203][INFO] - search_studies: Input Query includes geographical bounds. Inspect the results to ensure they match your intended region as one study can contain sites across various parts of the world.


Request URL: https://www.ncei.noaa.gov/access/paleo-search/study/search.json?dataPublisher=NOAA&NOAAStudyId=32833&limit=100


Parsing NOAA studies: 100%|██████████| 1/1 [00:00<00:00, 7084.97it/s]
[2026-04-06 14:11:19,554][INFO] - Retrieved 1 studies.


,StudyID,XMLID,StudyName,DataType,EarliestYearBP,MostRecentYearBP,EarliestYearCE,MostRecentYearCE,"Coverage [S, N, W, E]",StudyNotes,ScienceKeywords,Investigators,Publications,Sites,Funding
0,32833,74513,"Greater Yellowstone Ecoregion 1,250 Year Maxim...",CLIMATE RECONSTRUCTIONS,1180,-69,770,2019,"(44.0, 45.0, -111.0, -110.0)",Reconstruction of late-summer maximum air temp...,[Air Temperature Reconstruction],"Karen King, Maegen Rochner, Grant Harley","[{'Author': 'Karen J. Heeter, Maegen L. Rochne...","[[{'DataTableID': '45797', 'DataTableName': 'Y...",[]


The one item to notice on the summary is the geographical coordinates, which indicate an area rather than a single point. Let's get more information about the sites:

In [21]:
display(ds2.get_sites())

,DataTableID,DataTableName,TimeUnit,FileURL,Variables,FileDescription,TotalFilesAvailable,SiteID,SiteName,LocationName,Latitude,Longitude,MinElevation,MaxElevation,StudyID,StudyName
0,45797,Yellowstone2021MaxTemp,CE,https://www.ncei.noaa.gov/pub/data/paleo/treer...,"[age_CE, MaxTemp, MaxTemp-, MaxTemp+]",NOAA Template File,1,58871,Greater Yellowstone Ecoregion,Continent>North America>United States Of America,44,45,None,None,32833,"Greater Yellowstone Ecoregion 1,250 Year Maxim..."
